# AICE Associate 모의고사 (14문항)
**시나리오: 통신사 고객 이탈 예측 (이진 분류)**

당신은 통신사 데이터 분석가입니다. 고객 정보를 바탕으로 **해지(churn=1) 여부를 예측하는 모델**을 만들어야 합니다.

**푸는 방법**
1. 맨 위 `0. 준비` 셀만 그냥 실행 (연습용 데이터 파일 생성)
2. 문제 읽기 → 빈 셀에 직접 코드 작성 → 실행
3. 막히면 **👉 정답 보기** 클릭해서 확인 후 따라 치기
4. 실제 시험처럼 변수명(df, df1, X_train 등)을 문제에서 시키는 대로 쓰는 게 중요해요

| 컬럼 | 설명 |
|---|---|
| customerID | 고객 ID |
| gender | 성별 |
| age | 나이 (결측 있음) |
| region | 지역 |
| contract | 약정 유형 (월단위/1년약정/2년약정) |
| tenure | 가입 개월 수 |
| monthly_fee | 월 요금 (이상치 있음) |
| data_usage_gb | 월 데이터 사용량 (결측 있음) |
| cs_calls | 고객센터 문의 횟수 |
| payment | 결제 수단 |
| **churn** | **이탈 여부 (타깃, 0/1)** |

## 0. 준비 (문제 아님, 실행만 하세요)

In [ ]:
import numpy as np, pandas as pd
rng = np.random.default_rng(7)
n = 3000
contract = rng.choice(['월단위','1년약정','2년약정'], n, p=[0.5,0.3,0.2])
tenure = np.where(contract=='월단위', rng.integers(1,36,n), rng.integers(6,72,n))
fee = np.round(rng.normal(55000,15000,n),-2)
logit = -1.2 + 1.3*(contract=='월단위') - 0.03*tenure + 0.00002*(fee-55000)
cs = rng.poisson(1.2,n); logit = logit + 0.35*cs
pay = rng.choice(['카드','계좌이체','휴대폰결제'], n, p=[0.5,0.3,0.2]); logit = logit + 0.5*(pay=='휴대폰결제')
churn = (rng.random(n) < 1/(1+np.exp(-logit))).astype(int)
df = pd.DataFrame({
 'customerID':[f'C{100000+i}' for i in range(n)],
 'gender':rng.choice(['남','여'],n),
 'age':rng.integers(19,75,n).astype(float),
 'region':rng.choice(['서울','경기','부산','기타'],n,p=[0.35,0.3,0.1,0.25]),
 'contract':contract,'tenure':tenure,'monthly_fee':fee,
 'data_usage_gb':np.round(rng.gamma(2,8,n),1),'cs_calls':cs,'payment':pay,'churn':churn})
df.loc[rng.choice(n,150,replace=False),'age']=np.nan
df.loc[rng.choice(n,40,replace=False),'data_usage_gb']=np.nan
df.loc[rng.choice(n,12,replace=False),'monthly_fee']=-1
df.to_csv('churn_data.csv',index=False)

print('churn_data.csv 생성 완료')

## 1. 라이브러리 임포트

데이터 분석에 필요한 라이브러리를 임포트하세요.
- pandas는 `pd`, numpy는 `np`
- seaborn은 `sns`, matplotlib.pyplot은 `plt`

In [ ]:
# 여기에 직접 작성해 보세요

<details><summary><b>👉 정답 보기</b></summary>

```python
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
```

시험 첫 문제는 거의 항상 이거예요. 별칭(alias)까지 정확히 맞춰야 채점됩니다.
</details>

## 2. 데이터 불러오기

`churn_data.csv` 파일을 읽어 데이터프레임 **`df`** 에 저장하고, 앞 5행과 컬럼 정보를 확인하세요.

> 💡 힌트: pd.read_csv

In [ ]:
# 여기에 직접 작성해 보세요

<details><summary><b>👉 정답 보기</b></summary>

```python
df = pd.read_csv('churn_data.csv')
df.head()
df.info()
```

시험에서는 json이 나오기도 해요. 그땐 `pd.read_json('파일.json')`. `info()`로 결측치 개수와 자료형(object인지 숫자인지)을 먼저 보는 습관을 들이세요.
</details>

## 3. 시각화 (1) 타깃 분포

seaborn의 **countplot** 으로 `churn` 컬럼의 값별 개수를 그래프로 그리세요.

그래프를 보고 **이탈(1) 고객이 비이탈(0)보다 많은지 적은지** 답해 보세요.

In [ ]:
# 여기에 직접 작성해 보세요

<details><summary><b>👉 정답 보기</b></summary>

```python
sns.countplot(data=df, x='churn')
plt.show()

df['churn'].value_counts()
```

답: **적다** (0이 약 2,000명, 1이 약 1,000명).

타깃이 불균형하면 나중에 train_test_split에서 `stratify=y`를 꼭 써야 한다는 신호예요. 연속형 변수 분포를 물으면 `sns.histplot(data=df, x='tenure')`를 씁니다.
</details>

## 4. 시각화 (2) 상관관계

숫자형 컬럼들만 골라 **상관계수 heatmap** 을 그리세요. (annot=True)

`churn`과 **절댓값 기준 상관이 가장 큰 변수**는 무엇인가요?

In [ ]:
# 여기에 직접 작성해 보세요

<details><summary><b>👉 정답 보기</b></summary>

```python
corr = df.select_dtypes(include='number').corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm')
plt.show()

corr['churn'].sort_values()
```

답: **tenure** (음의 상관. 오래 쓴 고객일수록 덜 해지함).

`df.corr()`를 그냥 쓰면 문자열 컬럼 때문에 에러가 날 수 있어서 `select_dtypes(include='number')` 또는 `df.corr(numeric_only=True)`를 써요. 시험에서 '가장 상관이 높은 변수'를 묻는 객관식 형태로 자주 나옵니다.
</details>

## 5. 불필요한 컬럼 삭제

모델링에 의미 없는 `customerID` 컬럼을 삭제한 결과를 **`df1`** 에 저장하세요.

In [ ]:
# 여기에 직접 작성해 보세요

<details><summary><b>👉 정답 보기</b></summary>

```python
df1 = df.drop(columns=['customerID'])
df1.head()
```

`df.drop('customerID', axis=1)`도 같은 뜻이에요. axis=1이 컬럼, axis=0이 행. 원본 df를 건드리지 않고 새 변수에 저장하라는 조건을 꼭 지키세요.
</details>

## 6. 결측치 처리

`df1`의 결측치를 확인한 뒤 다음처럼 처리해 **`df2`** 에 저장하세요.
- `age` 결측치는 **중앙값(median)** 으로 채우기
- `data_usage_gb` 결측치가 있는 **행은 삭제**

In [ ]:
# 여기에 직접 작성해 보세요

<details><summary><b>👉 정답 보기</b></summary>

```python
df1.isnull().sum()

df2 = df1.copy()
df2['age'] = df2['age'].fillna(df2['age'].median())
df2 = df2.dropna(subset=['data_usage_gb'])

df2.isnull().sum()
```

채우기는 `fillna`, 삭제는 `dropna`. 특정 컬럼 기준으로만 삭제하려면 `subset=`을 씁니다.

`copy()`를 안 쓰면 SettingWithCopyWarning이 뜰 수 있어요. 마지막에 `isnull().sum()`이 전부 0인지 확인하면 끝.
</details>

## 7. 이상치 처리

`monthly_fee`(월 요금)가 **0 이하인 행**은 잘못 입력된 값입니다. 이 행들을 제거해 **`df3`** 에 저장하고, 몇 행이 남았는지 출력하세요.

In [ ]:
# 여기에 직접 작성해 보세요

<details><summary><b>👉 정답 보기</b></summary>

```python
df3 = df2[df2['monthly_fee'] > 0]
print(df3.shape)
```

답: **2,948행** (12행 제거).

조건 필터링은 `df[조건]`. 여러 조건은 `(조건1) & (조건2)` 처럼 괄호 필수. 특정 값을 바꾸라는 문제가 나오면 `df['col'].replace('기존','새값')`을 씁니다.
</details>

## 8. 원핫 인코딩

문자형(object) 컬럼 `gender`, `region`, `contract`, `payment` 를 **원핫 인코딩** 하세요. (`drop_first=True`)

결과를 **`df_preset`** 에 저장하세요.

In [ ]:
# 여기에 직접 작성해 보세요

<details><summary><b>👉 정답 보기</b></summary>

```python
cols = ['gender', 'region', 'contract', 'payment']
df_preset = pd.get_dummies(df3, columns=cols, drop_first=True)
df_preset.head()
```

AICE 단골 문제. `columns=`에 리스트로 넣는 것, `drop_first=True` 위치를 외워 두세요.

결과가 True/False로 나와도 정상이에요. 숫자로 보고 싶으면 `dtype=int`를 추가하면 됩니다.
</details>

## 9. 학습/검증 데이터 분리

`df_preset`에서 `churn`을 타깃 **y**, 나머지를 **X** 로 나누고, 다음 조건으로 분리하세요.
- 변수명: `X_train, X_test, y_train, y_test`
- 검증 데이터 비율 **20%**
- 타깃 비율 유지 (**stratify**)
- `random_state=42`

In [ ]:
# 여기에 직접 작성해 보세요

<details><summary><b>👉 정답 보기</b></summary>

```python
from sklearn.model_selection import train_test_split

X = df_preset.drop('churn', axis=1)
y = df_preset['churn']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)

print(X_train.shape, X_test.shape)
```

반환 순서 **X_train, X_test, y_train, y_test** 를 헷갈리면 이후 전부 틀려요. 꼭 외우세요.

`stratify=y`는 3번에서 본 불균형 때문에 필요한 옵션입니다.
</details>

## 10. 스케일링

`StandardScaler`로 X_train을 **fit_transform**, X_test는 **transform** 해서 다시 `X_train`, `X_test`에 저장하세요.

In [ ]:
# 여기에 직접 작성해 보세요

<details><summary><b>👉 정답 보기</b></summary>

```python
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)
```

**test에는 절대 fit 하지 않기**. 시험에서 가장 많이 틀리는 포인트예요.

MinMaxScaler를 쓰라고 하면 import만 `from sklearn.preprocessing import MinMaxScaler`로 바꾸면 사용법은 똑같아요.
</details>

## 11. 머신러닝 모델 학습

다음 두 모델을 만들고 학습하세요.
- **DecisionTreeClassifier**: `max_depth=5, random_state=42` → 변수명 **`dt`**
- **RandomForestClassifier**: `n_estimators=100, random_state=42` → 변수명 **`rf`**

In [ ]:
# 여기에 직접 작성해 보세요

<details><summary><b>👉 정답 보기</b></summary>

```python
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

dt = DecisionTreeClassifier(max_depth=5, random_state=42)
dt.fit(X_train, y_train)

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
```

회귀 문제면 `DecisionTreeRegressor`, `RandomForestRegressor`로 이름만 바뀌어요. 모듈 위치(tree, ensemble)도 같이 외우세요.

시험에서 LogisticRegression(`sklearn.linear_model`)이나 XGBoost, LightGBM이 나올 수도 있는데 사용법은 전부 `모델(옵션).fit(X_train, y_train)` 똑같아요.
</details>

## 12. 모델 성능 평가

두 모델로 `X_test`를 예측하고 **accuracy** 와 **classification_report** 를 출력하세요.

어느 모델의 정확도가 더 높은가요?

In [ ]:
# 여기에 직접 작성해 보세요

<details><summary><b>👉 정답 보기</b></summary>

```python
from sklearn.metrics import accuracy_score, classification_report

dt_pred = dt.predict(X_test)
rf_pred = rf.predict(X_test)

print('DT accuracy:', accuracy_score(y_test, dt_pred))
print('RF accuracy:', accuracy_score(y_test, rf_pred))
print(classification_report(y_test, rf_pred))
```

답: **RandomForest** (약 0.725 vs DecisionTree 약 0.724, 아주 근소한 차이).

인자 순서는 **(정답, 예측)** = `(y_test, pred)`.

회귀 문제면 `from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score` 를 써요. 결과 수치를 보고 '어떤 모델이 더 좋은가'를 고르는 문제가 자주 나옵니다.
</details>

## 13. 딥러닝 모델 만들고 학습

Keras로 DNN을 만들어 **`model`** 에 저장하고 학습하세요.
- Dense(64, relu) → Dropout(0.2) → Dense(32, relu) → Dropout(0.2) → **출력층**
- 출력층과 loss는 **이진 분류**에 맞게 직접 고르기
- optimizer=`adam`, metrics=`['accuracy']`
- **EarlyStopping**: `monitor='val_loss', patience=5`
- **ModelCheckpoint**: `'best_model.keras'`, `save_best_only=True`
- epochs=50, batch_size=32, `validation_data=(X_test, y_test)`
- 학습 결과를 **`history`** 에 저장

In [ ]:
# 여기에 직접 작성해 보세요

<details><summary><b>👉 정답 보기</b></summary>

```python
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

model = Sequential([
    Dense(64, activation='relu', input_shape=(X_train.shape[1],)),
    Dropout(0.2),
    Dense(32, activation='relu'),
    Dropout(0.2),
    Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam',
              loss='binary_crossentropy',
              metrics=['accuracy'])

es = EarlyStopping(monitor='val_loss', patience=5)
mc = ModelCheckpoint('best_model.keras', monitor='val_loss', save_best_only=True)

history = model.fit(X_train, y_train,
                    epochs=50, batch_size=32,
                    validation_data=(X_test, y_test),
                    callbacks=[es, mc])
```

**출력층과 loss 짝꿍은 무조건 외우기**

| 문제 유형 | 출력층 | loss |
|---|---|---|
| 이진 분류 | `Dense(1, activation='sigmoid')` | `binary_crossentropy` |
| 다중 분류 (y가 0,1,2...) | `Dense(클래스수, activation='softmax')` | `sparse_categorical_crossentropy` |
| 다중 분류 (y가 원핫) | `Dense(클래스수, activation='softmax')` | `categorical_crossentropy` |
| 회귀 | `Dense(1)` | `mse` (metrics는 `['mae']`) |

시험 환경 버전에 따라 체크포인트 파일명이 `.h5`로 나올 수도 있어요. 문제에 적힌 대로 쓰면 됩니다. `input_shape` 경고가 떠도 실행엔 문제 없어요.
</details>

## 14. 학습 곡선 시각화

`history`를 이용해 **loss와 val_loss** 를 한 그래프에 그리세요. (x축 Epochs, y축 Loss, 범례 포함)

In [ ]:
# 여기에 직접 작성해 보세요

<details><summary><b>👉 정답 보기</b></summary>

```python
plt.plot(history.history['loss'], label='loss')
plt.plot(history.history['val_loss'], label='val_loss')
plt.title('Model Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.show()
```

`history.history`는 딕셔너리예요. 키 이름은 `loss`, `val_loss`, `accuracy`, `val_accuracy`. 헷갈리면 `history.history.keys()`로 확인하면 됩니다.

val_loss가 다시 올라가기 시작하는 지점이 과적합 시작이고, EarlyStopping이 거기서 멈춰 줍니다.
</details>

---
# 🏁 마무리 체크리스트

**시험장에서 막힐 때 (제한적 오픈북)**
- 함수 이름 뒤에 `?` 붙여 실행: `pd.get_dummies?`
- 괄호 안에서 **Shift + Tab**: 인자 목록 보기
- `help(train_test_split)`

**자주 틀리는 것 TOP 5**
1. `train_test_split` 반환 순서 (X_train, X_test, y_train, y_test)
2. test에 `fit_transform` 써버리기 (transform만!)
3. 출력층 활성화 함수와 loss 짝 안 맞추기
4. 문제에서 지정한 **변수명** 안 지키기 (df1, df_preset 등)
5. `drop` 후 새 변수에 저장 안 하기

**다음 단계 연습 아이디어**
- 같은 데이터로 `monthly_fee`를 타깃으로 바꿔 **회귀 버전**으로 풀어보기 (Regressor, MAE, Dense(1), mse)
- 처음부터 정답 안 보고 타이머 60분 맞추고 다시 풀기